# VMamba-T — last-stage fine-tuning

The genuine authors' VMamba architecture is saved locally in `vmamba_official.py`; the final VMamba stage and the classifier are updated. Feature caching cannot be used because these features now change.


In [1]:
import sys
import torch
from pathlib import Path

MODEL_FOLDER = (
    Path.home() / "Desktop/Paper replication/model_reproductions_7_models"
    / "07_vmamba_fine_tuned"
)
sys.path.insert(0, str(MODEL_FOLDER))

from vit_training_helpers import configure_parameters, set_seed, train_fine_tuned_stable
from vmamba_loader import build_vmamba


In [2]:
set_seed(42)
model, last_stage = build_vmamba()
initial_checkpoint = (
    MODEL_FOLDER.parent / '06_vmamba_frozen/notebook_results/best_validation_accuracy.pth'
)
model.load_state_dict(torch.load(initial_checkpoint, map_location='cpu', weights_only=True))
configure_parameters(model, phase='fine_tuned', last_stage=last_stage)
print(last_stage)
print('Trainable:', sum(p.numel() for p in model.parameters() if p.requires_grad))


/Users/olzhi/Desktop/Paper replication/model_reproductions_7_models/07_vmamba_fine_tuned/vmamba_official.py:30: UserWarning: Triton not installed, fall back to pytorch implements.
  warnings.warn("Triton not installed, fall back to pytorch implements.")
/Users/olzhi/Desktop/Paper replication/model_reproductions_7_models/07_vmamba_fine_tuned/vmamba_official.py:854: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  def forward(ctx, u, delta, A, B, C, D=None, delta_bias=None, delta_softplus=False, oflex=True, backend=None):
/Users/olzhi/Desktop/Paper replication/model_reproductions_7_models/07_vmamba_fine_tuned/vmamba_official.py:869: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, dout, *args):


Sequential(
  (blocks): Sequential(
    (0): VSSBlock(
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (op): SS2D(
        (out_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (in_proj): Linear(in_features=768, out_features=768, bias=False)
        (act): SiLU()
        (conv2d): Conv2d(768, 768, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=768, bias=False)
        (x_proj): Linear(in_features=768, out_features=200, bias=False)
        (dt_projs): Linear(in_features=48, out_features=3072, bias=False)
        (out_act): Identity()
        (out_proj): Linear(in_features=768, out_features=768, bias=False)
        (dropout): Identity()
      )
      (drop_path): timm.DropPath(0.1846153885126114)
      (norm2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=3072, out

In [3]:
train_fine_tuned_stable(
    model=model, last_stage=last_stage,
    output_dir=MODEL_FOLDER/'notebook_results_cpu_notebook',
    epochs=10, learning_rate=1e-5, batch_size=1, center_crop=True,
)


{"epoch": 1, "train_loss": 0.31365749082893174, "train_accuracy": 0.8920454545454546, "validation_loss": 0.18502125187951607, "validation_accuracy": 0.9333333333333333}


{"epoch": 2, "train_loss": 0.1900589619199632, "train_accuracy": 0.9318181818181818, "validation_loss": 0.21761254419977075, "validation_accuracy": 0.9555555555555556}


{"epoch": 3, "train_loss": 0.26577632527445744, "train_accuracy": 0.9375, "validation_loss": 0.20402011709652976, "validation_accuracy": 0.9555555555555556}


{"epoch": 4, "train_loss": 0.2496536998407202, "train_accuracy": 0.9602272727272727, "validation_loss": 0.199927781365095, "validation_accuracy": 0.9777777777777777}


{"epoch": 5, "train_loss": 0.3217354118310087, "train_accuracy": 0.9488636363636364, "validation_loss": 0.1852371270091679, "validation_accuracy": 0.9777777777777777}


...

{"epoch": 7, "train_loss": 0.2051488756767992, "train_accuracy": 0.9715909090909091, "validation_loss": 0.21963132836018545, "validation_accuracy": 0.9777777777777777}


{"epoch": 8, "train_loss": 0.23207923847799353, "train_accuracy": 0.9602272727272727, "validation_loss": 0.20253849064632368, "validation_accuracy": 0.9777777777777777}


{"epoch": 9, "train_loss": 0.17751089242523338, "train_accuracy": 0.9602272727272727, "validation_loss": 0.15989266666795285, "validation_accuracy": 0.9777777777777777}


{"epoch": 10, "train_loss": 0.261292226838048, "train_accuracy": 0.9602272727272727, "validation_loss": 0.1753210461649115, "validation_accuracy": 0.9777777777777777}


{
  "final": {
    "validation_loss": 0.1753210461649115,
    "validation_accuracy": 0.9777777777777777,
    "correct_predictions": 44,
    "incorrect_predictions": 1,
    "prediction_count": 45,
    "confusion_matrix": [
      [
        23,
        1
      ],
      [
        0,
        21
      ]
    ],
    "Low": {
      "precision": 1.0,
      "recall": 0.9583333333333334,
      "f1-score": 0.9787234042553191,
      "support": 24.0
    },
    "High": {
      "precision": 0.9545454545454546,
      "recall": 1.0,
      "f1-score": 0.9767441860465116,
      "support": 21.0
    },
    "macro_f1": 0.9777337951509153,
    "weighted_f1": 0.9777997690912089
  },
  "best_loss": {
    "validation_loss": 0.15989266666795285,
    "validation_accuracy": 0.9777777777777777,
    "correct_predictions": 44,
    "incorrect_predictions": 1,
    "prediction_count": 45,
    "confusion_matrix": [
      [
        23,
        1
      ],
      [
        0,
        21
      ]
    ],
    "Low": {
      "preci